In [ ]:
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
import daaa as daaa

In [ ]:
import importlib as il

In [ ]:
il.reload(daaa)

In [ ]:
jr = sio.loadmat("/home/josephg/Documents/EDAA_unmixing/JasperRidge.mat")
jasp = jr['Y'] / np.sum(jr['Y']**2, axis=0)**(1/4)

In [ ]:
da1 = daaa.DAAA(n_components=4, initial_regularization=0.0, time_constant=5, delta=0, epochs=1, mu=0)
da1.PPA=False

In [ ]:
daaa.DAAA(n_components=4, initial_regularization=0.0, time_constant=5, delta=0, epochs=1, mu=0)
da1._initialize(jasp)

In [ ]:
da1.H.shape

In [ ]:
da1.train(jasp)

In [ ]:
fig, ax = plt.subplots(1,4, figsize=(15,5))
for i in range(4):
    ax[i].imshow(da1.H[i].reshape(100,100))

In [ ]:
da1.train(jasp)

In [ ]:
fig, ax = plt.subplots(1,4, figsize=(15,5))
for i in range(4):
    ax[i].imshow(da1.H[i].reshape(100,100))

In [ ]:
da1.train(jasp)

In [ ]:
fig, ax = plt.subplots(1,4, figsize=(15,5))
for i in range(4):
    ax[i].imshow(da1.H[i].reshape(100,100))

In [ ]:
da1.train(jasp)

In [ ]:
fig, ax = plt.subplots(1,4, figsize=(15,5))
for i in range(4):
    ax[i].imshow(da1.H[i].reshape(100,100))

In [ ]:
da1.train(jasp)

In [ ]:
fig, ax = plt.subplots(1,4, figsize=(15,5))
for i in range(4):
    ax[i].imshow(da1.H[i].reshape(100,100))

In [ ]:
g1 = da1.H>0
gprod= np.prod(da1.H.astype(np.float128), axis=0)
plt.imshow(gprod.reshape(100,100))
plt.title(gprod.sum())
plt.colorbar()

In [ ]:
da1.mu = 1000

In [ ]:
da1.train(jasp)

In [ ]:
g1 = da1.H>0
gprod= np.prod(da1.H.astype(np.float128), axis=0)
plt.imshow(gprod.reshape(100,100))
plt.title(gprod.sum())
plt.colorbar()

In [ ]:
fig, ax = plt.subplots(1,4, figsize=(15,5))
for i in range(4):
    ax[i].imshow(da1.H[i].reshape(100,100))

In [ ]:
g1 = da1.H>0
gprod= np.prod(da1.H.astype(np.float128), axis=0)
plt.imshow(gprod.reshape(100,100))
plt.title(gprod.sum())
plt.colorbar()

In [ ]:
da1.mu = 2000
da1.train(jasp)

In [ ]:
g1 = da1.H>0
gprod= np.prod(da1.H.astype(np.float128), axis=0)
plt.imshow(gprod.reshape(100,100))
plt.title(gprod.sum())
plt.colorbar()

In [ ]:
da1.mu = 2000
da1.train(jasp)

In [ ]:
fig, ax = plt.subplots(1,4, figsize=(15,5))
for i in range(4):
    ax[i].imshow(da1.H[i].reshape(100,100))

In [ ]:
g1 = da1.H>0
gprod= np.prod(da1.H.astype(np.float128), axis=0)
plt.imshow(gprod.reshape(100,100))
plt.title(gprod.sum())
plt.colorbar()

In [ ]:
da1.mu = 2000
da1.train(jasp)

In [ ]:
g1 = da1.H>0
gprod= np.prod(da1.H.astype(np.float128), axis=0)
plt.imshow(gprod.reshape(100,100))
plt.title(gprod.sum())
plt.colorbar()

In [ ]:
fig, ax = plt.subplots(1,4, figsize=(15,5))
for i in range(4):
    ax[i].imshow(da1.H[i].reshape(100,100))

In [ ]:
da1.mu = 1
da1.train(jasp)

In [ ]:
da1.mu = 0.5
da1.train(jasp)

In [ ]:
g1 = da1.H>0
gprod= np.prod(da1.H.astype(np.float128), axis=0)
plt.imshow(gprod.reshape(100,100))
plt.title(gprod.sum())
plt.colorbar()

In [ ]:
fig, ax = plt.subplots(1,4, figsize=(15,5))
for i in range(4):
    ax[i].imshow(da1.H[i].reshape(100,100))

In [ ]:
def e_reg_max(aa, data):
    #only calculate the main portion of the objective function
    obj = daaa.objective(data.T, aa.W, aa.H, 0, aa.weights)
    denom = np.sum(aa.H**2, axis=0).mean() - 1/len(aa.H)
    return obj / denom

In [ ]:
e_reg_max(da1, jasp)

In [ ]:
#f mpp could be moved to a 'utilities' file
def mpp(S, weights):
    valid = weights >0
    return 1-(S[:,valid]==1).sum()/valid.sum()
    

In [ ]:
mpp(da1.H, da1.weights)

In [ ]:
da1.W.shape

In [ ]:
boundary_sparsify(da1, jasp)

In [ ]:
def product_sum(aa):
    return (1-np.sum(aa.H**2*(3-2*aa.H), axis=0))/6

def boundary_sparsify(aa, data, n_runs=10):
    EPS = 1e-3
    regularization_increment = e_reg_max(aa, data)
    psum = product_sum(aa).sum()/len(data)
    aa.mu = 0
    while psum > EPS:
        aa.mu += regularization_increment
        for i in range(n_runs):
            switch_training(aa, data)
        psum = product_sum(aa).sum()/len(data)
        print("psum is ", psum, " while regularization is ",aa.mu)




def sparsify(aa, data, mpp_tol, n_runs=10):
    step_delta = e_reg_max(aa, data) / n_runs
    initial_mpp = mpp(aa.H, aa.weights)
    scaling_factor_base = mpp_tol**(1/n_runs)
    #need to set reg_max!
    reg_max = e_reg_max(aa, data)
    
    go_on = True
    while go_on:
        sparsity_sweep(aa, data, step_delta, reg_max)

        eval_mpp = mpp(aa.H, aa.weights)
        print(reg_max, " mpp is ", eval_mpp)
        go_on = eval_mpp > mpp_tol

        if initial_mpp > eval_mpp*1.5:
            reg_max *= 1.1
        elif initial_mpp > eval_mpp*1.3:
            reg_max *= 1.3
        elif initial_mpp > eval_mpp*1.1:
            reg_max *= 1.5
        else:
            reg_max *= 2
        step_delta = reg_max / n_runs

def switch_training(aa, data):
        aa.update_W_AA(data)
        aa.update_all_abundances_BCD(data)
        aa.append_record(data.T)

def sparsity_sweep(aa, data, step_delta, reg_max):
    aa.mu = 0
    aa.T = 0 # note that a different sign convention is used here than in bluth.py
    # would love to change that!

    while aa.T > -reg_max:
        switch_training(aa, data)
        aa.T -= step_delta
    aa.T = 0


def shake(aa, data, n_runs=10):
    objs = []
    if aa.record:
        record = True
    else:
        record = False
        aa.obj_rec = []
    aa.record = True

    aa.mu=0
    aa.T=0
    for i in range(n_runs):
        switch_training(aa, data)
    
    

    objs = np.array([aa.obj_rec[-i][1] for i in range(1,n_runs+1)])
    obj_mean = objs.mean()
    obj_max = objs[:n_runs//2].max()
    obj_std = objs.std()

    bound = obj_mean
    gain = obj_mean / (np.sum(aa.H**2, axis=0).mean()-1/len(aa.H))

    go_on = True
    i = 0
    while go_on:
        sparsity_alternation(aa, data, reg_level=gain, n_runs=10)
        print("Now it equiliberates", gain)
        for i in range(n_runs):
            switch_training(aa, data)
        objs = np.array([aa.obj_rec[-i][1] for i in range(1,n_runs+1)])
        obj_min = objs.min()
        go_on = obj_min < bound
        if bound > objs.mean():
            bound = objs.mean()
        if i > n_runs**2:
            go_on = False

        print(objs)
        print(obj_min, bound)
        if go_on:
            gain += objs.mean() / (np.sum(aa.H**2, axis=0).mean()-1/len(aa.H)) 


    if not record:
        aa.record = False
    aa.T = 0


def sparsity_alternation(aa, data, reg_level, n_runs):
    aa.mu = 0
    aa.T = 0 # note that a different sign convention is used here than in bluth.py
    # would love to change that!

    i = 0
    while i < n_runs:
        if i%2 == 0:
            aa.T = - reg_level
        else:
            aa.T = 0
        switch_training(aa, data)
        i += 1
    aa.T = 0


class Node():
    def __init__(self, classifier, spatial_map=()):
        if len(spatial_map) > 0:
            self.map = spatial_map.astype(np.float16)
        else:
            self.map = ()
        self.classifier = classifier


class EHU():
    '''
    expansive hierarchical unmixing 
    '''
    def __init__(self, n_runs = 10, record=False):
        self.n_runs = 10
        self.record = record
        self.obj_rec = []
        self.sc_f = 2 #scaling_factor
        self.mu = 0
        self.gamma = 0


    def get_depth(self):
        my = list(self.nodes.keys())
        my.sort(key=len)
        self.depth = len(my[-1])
        return self.depth

    
    def expand_spectra(self):
        L = self.get_depth()
        base_nodes = [n for n in self.nodes if len(n)==L]
        spectra = np.zeros((len(self.nodes[''].classifier),len(base_nodes)), dtype=np.float32)
        for i,n in enumerate(base_nodes):
            bns = [self.sc_f*l * self.nodes[n[:l]].classifier for l in range(1,L+1)]
            bn_concat = np.concatenate(bns)
            spectra[:,i] = bn_concat
        return spectra, base_nodes
    
    def get_maps_at_level(self, l):
        L = self.get_depth()
        nodes_at_level = [n for n in self.nodes if len(n)==l]
        size = len(nodes_at_level)
        S = np.zeros((size,len(self.nodes[nodes_at_level[0]].map)), np.float16)
        lamlab = {}
        i = 0 
        for n in self.nodes:
            if len(n) == l:                
                lamlab[n] = i
                i += 1
        for n in self.nodes:
            if len(n) == L:
                #print(S.shape, self.nodes[n].map.shape)
                S[lamlab[n[:l]]] += self.nodes[n].map

        return S, lamlab

    def expand_data(self, data):
        L = self.get_depth()
        expanded_data = np.array([self.sc_f*l * data for l in range(1,L+1)]).reshape(L*data.shape[0], data.shape[1])
        return expanded_data
        
    def spatial_update(self, data):
        #get spectra
        W, Wlabs = self.expand_spectra()

        #get_original_map
        H, Hlabs = self.get_maps_at_level(self.get_depth())

        aa = daaa.DAAA(n_components=len(H), initial_regularization=0.0, 
                       time_constant=1, delta=0, epochs=1, mu=0)
        aa.W = W
        aa.H = H
        aa.weights = np.ones(data.shape[1])
        aa.T = self.gamma
        aa.mu = self.mu
        aa.update_all_abundances_BCD(self.expand_data(data))
        for i,n in enumerate(Hlabs):
            self.nodes[n].map = aa.H[Hlabs[n],:]

    
    def spectral_update(self, data, init=False):
        L = self.get_depth()
        for l in range(1,L+1):
            H, Hlab = self.get_maps_at_level(l)
            W = np.array([self.nodes[n].classifier for n in Hlab]).T
            aa = daaa.DAAA(n_components=len(H), initial_regularization=0.0, 
                       time_constant=1, delta=0, epochs=1, mu=0)
            aa.W = W
            aa.H = H
            aa.weights = np.ones(data.shape[1])
            aa.T = self.gamma
            aa.mu = self.mu
            if init:
                aa.PPA = True
            aa.update_W_AA(data)
            for i,n in enumerate(Hlab):
                self.nodes[n].classifier = aa.W[:,i]


    def get_end_nodes(self):
        end_nodes = [i for i in self.nodes if len(i)==d]
        return end_nodes

    def grow_node(self, to_grow):
        d = self.get_depth()
        self.nodes[to_grow +'1'] = Node(spatial_map=np.array([]),
                                            classifier= copy.deepcopy(self.nodes[to_grow].classifier))
        if len(to_grow)==d:
            end_nodes = self.get_end_nodes()
            self.nodes[to_grow].map /= 2
            for n in end_nodes:
                self.nodes[n+'0'] = Node(spatial_map=self.nodes[n].map,
                                         classifier=copy.deepcopy(self.nodes[n].classifier))
                self.nodes[n].map = np.array([])
            
        
        if len(to_grow)<(d-1):
            n_to_add = to_grow + '10'
            while len(n_to_add) <= d:
                self.nodes[n_to_add] = Node(spatial_map=self.nodes[to_grow].map, 
                                            classifier = copy.deepcopy(self.nodes[to_grow+'1'].classifier_r))
                self.nodes[n_to_add].classifier_n = copy.deepcopy(self.nodes[to_grow+'1'].classifier_n)
                n_to_add += '0'
        else:
            self.nodes[to_grow +'0'].classifier = copy.deepcopy(self.nodes[to_grow].classifier)
            self.nodes[to_grow + '0'].map /=2
            self.nodes[to_grow + '1'].map = copy.deepcopy(self.nodes[to_grow+'0'].map)

        
        if self._use_norm:
            for n in self.nodes:
                self.nodes[n].classifier = copy.deepcopy(self.nodes[n].classifier_n)
                
        self.nodes[to_grow].splitter = [0*self.nodes[to_grow].classifier,0]


        
    def initialize_initial_nodes(self, data):
        self.nodes = {
            '0':Node(spatial_map = 0.5*np.ones(data.shape[-1], dtype=np.float16),
                    classifier = data.mean(axis=1)),
            '1':Node(spatial_map = 0.5*np.ones(data.shape[-1], dtype=np.float16),
                    classifier = data.mean(axis=1)),
             '':Node(classifier = data.mean(axis=1)) 
        }
        self.spectral_update(data, init=True)
        self.spatial_update(data)
        #self.aa = daaa.DAAA(n_components=2, initial_regularization=0.0, 
        #               time_constant=n_runs, delta=0, epochs=1, mu=0)
        #self.aa._initialize(data)
    #should have a list of nodes akin to BLUTH
    #however, the spatial information should persist at only the lowest level

    #contains a daaa sub-class, re-generated each time 

    def train(self, data):
        for i in range(self.n_runs):
            self.spectral_update(data)
            self.spatial_update(data)
            


def SMUG(ehu, data, endmember_target):
    '''
    '''
    #initialize with 2 endmembers

    #equiliberate

    #while the convergence criteria is not met
        #iterate over endmembers

 

    



In [ ]:
ehu = EHU()

In [ ]:
ehu.initialize_initial_nodes(jasp)

In [ ]:
ehu.sc_f=2

In [ ]:
plt.plot(ehu.nodes[''].classifier)
plt.plot(ehu.nodes['0'].classifier)
plt.plot(ehu.nodes['1'].classifier)

In [ ]:
ehu.train(jasp)

In [ ]:
plt.imshow(ehu.nodes['0'].map.reshape(100,100))
plt.colorbar()

In [ ]:
plt.imshow(ehu.nodes['1'].map.reshape(100,100))
plt.colorbar()

In [ ]:
sparsity_sweep(da1, jasp, 500, 5000)

In [ ]:
mpp(da1.H, da1.weights)

In [ ]:
tuples = [[i for i in range(4) if i != j] for j in range(4)]

In [ ]:
np.prod(da1.H.astype(np.float128)[tuples[0], :], axis=0).shape

In [ ]:
indGsum = (1-np.sum(da1.H**2*(3-2*da1.H), axis=0))/6

In [ ]:
indGsum = (1-np.sum(da1.H**2*(3-2*da1.H), axis=0))/6
g1 = da1.H>0
#gprod= np.sum([np.prod(da1.H.astype(np.float128)[tuples[i], :], axis=0) for i in range(4)], axis=0)
plt.imshow(indGsum.reshape(100,100))
plt.title(indGsum.sum())
plt.colorbar()

In [ ]:
g1 = da1.H>0
gprod= np.sum([np.prod(da1.H.astype(np.float128)[tuples[i], :], axis=0) for i in range(4)], axis=0)
plt.imshow(indGsum.reshape(100,100))
plt.title(indGsum.sum())
plt.colorbar()

In [ ]:
g1 = da1.H>0
gprod= np.sum([np.prod(da1.H.astype(np.float128)[tuples[i], :], axis=0) for i in range(4)], axis=0)
plt.imshow(gprod.reshape(100,100))
plt.title(gprod.sum())
plt.colorbar()

In [ ]:
fig, ax = plt.subplots(1,4, figsize=(15,5))
for i in range(4):
    ax[i].imshow(da1.H[i].reshape(100,100))

In [ ]:
da1.obj_rec

In [ ]:
g1 = da1.H>0
gsum = g1.sum(axis=0)
plt.imshow(gsum.reshape(95,95))
plt.colorbar()

In [ ]:
fig, ax = plt.subplots(1,3)
for i in range(3):
    ax[i].imshow(da1.H[i].reshape(95,95))

In [ ]:
da1.train(sy)

In [ ]:
g1 = da1.H>0
gsum = g1.sum(axis=0)
plt.imshow(gsum.reshape(95,95))
plt.colorbar()

In [ ]:
da1.mu=0.5

In [ ]:
da1.train(sy)

In [ ]:
g1 = da1.H>0
gsum = g1.sum(axis=0)
plt.imshow(gsum.reshape(95,95))
plt.colorbar()

In [ ]:
da1.train(sy)
g1 = da1.H>0
gsum = g1.sum(axis=0)
plt.imshow(gsum.reshape(95,95))
plt.colorbar()

In [ ]:
da1.train(sy)
g1 = da1.H>0
gsum = g1.sum(axis=0)
plt.imshow(gsum.reshape(95,95))
plt.colorbar()

In [ ]:
gprod = da1.H[0]*da1.H[1]*da1.H[2]

In [ ]:
plt.imshow(gprod.reshape(95,95)*9)
plt.colorbar()

In [ ]:
plt.imshow(da1.H[0].reshape(95,95))
plt.colorbar()

In [ ]:
a2 = (da1.H[1]**2 + da1.H[2]**2)/(da1.H[1]+da1.H[2]+1e-32)**2
a3 = (da1.H[1]**3 + da1.H[2]**3)/(da1.H[1]+da1.H[2]+1e-32)**3

In [ ]:
plt.imshow(a2.reshape(95,95))
plt.colorbar()

In [ ]:
plt.imshow(a3.reshape(95,95))
plt.colorbar()

In [ ]:
plt.imshow((a2-a3).reshape(95,95))
plt.colorbar()

In [ ]:
x = np.argmax(a2-a3)

In [ ]:
x

In [ ]:
a2[x]

In [ ]:
a3[x]

In [ ]:
xval = np.arange(0,1,.001)

In [ ]:
xval

In [ ]:
p = 2*(1-a3[x])*xval**3 - 3*(1+a2[x]-2*a3[x])*xval**2 + 6*(a2[x]-a3[x])*xval+1-3*a2[x]+2*a3[x]

In [ ]:
pl = lambda y: 2*(1-a3[x])*y**3 - 3*(1+a2[x]-2*a3[x])*y**2 + 6*(a2[x]-a3[x])*y+1-3*a2[x]+2*a3[x]
pl1 = lambda y: 1-a2[x]*(1-y)**2-y**2-a2[x]

In [ ]:
plt.plot(xval, p)
plt.scatter(da1.H[0,x], pl(da1.H[0,x]))
plt.plot(xval, pl1(xval))

In [ ]:
y0 = da1.H[0,x]

In [ ]:
plt.plot(xval, p)
plt.scatter(da1.H[0,x], pl(da1.H[0,x]))

In [ ]:
linfit = lambda y: (6*(1-a3[x])*y0**2-6*(1+a2[x]-2*a3[x])*y0+6*(a2[x]-a3[x]))*(y-y0)+pl(da1.H[0,x])

In [ ]:
plt.plot(xval, p)
plt.scatter(da1.H[0,x], pl(da1.H[0,x]))
plt.plot(xval, linfit(xval))

In [ ]:
1-a3[x]

In [ ]:
max_curve=12*(1-a3[x])-6-6*a2[x]+12*a3[x]

In [ ]:
max_curve

In [ ]:
linfit = lambda y: (6*(1-a3[x])*y0**2-6*(1+a2[x]-2*a3[x])*y0+6*(a2[x]-a3[x]))*(y-y0)+pl(da1.H[0,x])
majorizer = lambda y: linfit(y)+max_curve/2*(y-y0)**2

In [ ]:
plt.plot(xval, p)
plt.scatter(da1.H[0,x], pl(da1.H[0,x]))
plt.plot(xval, linfit(xval))
plt.plot(xval, majorizer(xval))

In [ ]:
old = lambda y: -y**2
new = lambda y: -y**2*(3-2*y)

In [ ]:
plt.plot(xval, old(xval))
plt.plot(xval, new(xval))

In [ ]:
s_not = (da1.H[1,x]*da1.W[:,1]+da1.H[2,x]*da1.W[:,2])/(da1.H[1,x]+da1.H[2,x])

In [ ]:
plt.plot(s_not)
plt.plot(da1.W[:,0])

In [ ]:
(sy[:,x]-.5*da1.W[:,0]-(1-.5)*s_not).T@(sy[:,x]-.5*da1.W[:,0]-(1-.5)*s_not)

In [ ]:
ev = lambda y: (sy[:,x]-y*da1.W[:,0]-(1-y)*s_not).T@(sy[:,x]-y*da1.W[:,0]-(1-y)*s_not)

In [ ]:
err = (sy[:,x]-s_not).T@(da1.W[:,0]-s_not)
recap = (da1.W[:,0]-s_not).T@(da1.W[:,0]-s_not)

In [ ]:
recap = (da1.W[:,0]-s_not).T@(da1.W[:,0]-s_not)

In [ ]:
recap

In [ ]:
e_min = err/recap

In [ ]:
e_min

In [ ]:
e_p = lambda y: -err + recap*y

In [ ]:
plt.plot(xval, [ev(i) for i in xval])

In [ ]:
plt.plot(xval, e_p(xval))

In [ ]:
da1.H[:,x]

In [ ]:
plt.plot(sy[:,x])
plt.plot(s_not)
plt.plot(da1.W[:,0])
plt.plot(da1.W[:,0]*0.72 + (1-0.72)*s_not)

In [ ]:
plt.plot(xval, p)
plt.scatter(da1.H[0,x], pl(da1.H[0,x]))
plt.plot(xval, linfit(xval))
plt.plot(xval, majorizer(xval))
plt.scatter(m_min, majorizer(m_min))

In [ ]:
linfit = lambda y: (6*(1-a3[x])*y0**2-6*(1+a2[x]-2*a3[x])*y0+6*(a2[x]-a3[x]))*(y-y0)+pl(da1.H[0,x])
majorizer = lambda y: linfit(y)+max_curve/2*(y-y0)**2

In [ ]:
m_min = (max_curve*y0-(6*(1-a3[x])*y0**2-6*(1+a2[x]-2*a3[x])*y0+6*(a2[x]-a3[x])))/max_curve

In [ ]:
m_min

In [ ]:
np.arange(3)

In [ ]:
#Need to setup code for automatically calculating adjustments only with knowledge of opposites
endmember = 0 #set as iterate starts
oppo_list = [i for i in range(len(da1.H)) if i is not endmember]
S = da1.H
a1 = np.array([S[i]+1e-32 for i in oppo_list]) 
a1 /= np.sum(a1, axis=0)
a_x = lambda x: (np.sum(a1**x, axis=0))#/(np.sum(S[oppo_list], axis=0)+1e-16)**x
a2 = a_x(2)
a3 = a_x(3)

In [ ]:
def calc_oppo_L_a1(endmember, S, spectra):
    oppo_list = [i for i in range(len(S)) if i is not endmember]
    L = S.shape[-1]
    a1 = np.array([S[i]+1e-8 for i in oppo_list]) 
    a1 /= np.sum(a1, axis=0)
    return oppo_list, L, a1

def calc_m_adjustments(endmember, S, spectra):
    oppo_list, _, a1 = calc_oppo_L_a1(endmember, S, spectra)
    a_x = lambda x: (np.sum(a1**x, axis=0))
    a2 = a_x(2)
    a3 = a_x(3)

    mx_curve = 12*(1-a3)-6-6*a2+12*a3 #numbers come from reg.
    y0 = S[endmember]
    lin =  (6*(1-a3)*y0**2-6*(1+a2-2*a3)*y0+6*(a2-a3))

    m_numerator = mx_curve*y0-lin
    m_denominator = mx_curve

    return m_numerator, m_denominator

def calc_err_grad(endmember, S, spectra, data):
    oppo_list, L, a1 = calc_oppo_L_a1(endmember, S, spectra)
    conv_spec = np.sum([np.outer(a1[i], spectra[:,oppo_list[i]]) for i in range(len(oppo_list))], axis=0)

    basic_numerator = ((data.T-conv_spec).reshape(L,1,-1)@((spectra[:,endmember])-conv_spec).reshape(L,-1,1)).reshape(L)
    delta = (spectra[:,endmember]-conv_spec)
    basic_denominator = (delta.reshape(L,1,-1)@delta.reshape(L,-1,1)).reshape(L)
    
    return basic_numerator, basic_denominator

def make_mu_adj(basic_numerator, basic_denominator, m_numerator, m_denominator):
    return lambda x: np.minimum(1,
                                np.maximum((basic_numerator + x * m_numerator)/(basic_denominator + x * m_denominator),0))

In [ ]:
m_n, m_d = calc_m_adjustments(1, da1.H, da1.W)
b_n, b_d = calc_err_grad(1, da1.H, da1.W, sy)

In [ ]:
mu1 = make_mu_adj(b_n, b_d, m_n, m_d)

In [ ]:
plt.imshow((da1.H[1]).reshape(95,95))
plt.colorbar()

In [ ]:
plt.imshow((mu1(0.0)).reshape(95,95))
plt.colorbar()

In [ ]:
plt.imshow((mu1(.1)-mu1(0.0)).reshape(95,95))
plt.colorbar()

In [ ]:
m_n, m_d = calc_m_adjustments(2, da1.H, da1.W)
b_n, b_d = calc_err_grad(2, da1.H, da1.W, sy)
mu2 = make_mu_adj(b_n, b_d, m_n, m_d)
plt.imshow((mu2(.1)-mu2(0.0)).reshape(95,95))
plt.colorbar()

In [ ]:
mx_curve = 12*(1-a3)-6-6*a2+12*a3
y0 = S[endmember]

In [ ]:
lin =  (6*(1-a3)*y0**2-6*(1+a2-2*a3)*y0+6*(a2-a3))

In [ ]:
plt.imshow(mx_curve.reshape(95,95))

In [ ]:
plt.imshow((lin).reshape(95,95))
plt.colorbar()

In [ ]:
#needs to be set before updating
mu = 1

In [ ]:
# calculate converse spectra
s_con = np.sum([S[i]*a1 for i in range(len(S)) if i is not endmember], axis=0)

In [ ]:
# calculate converse spectra
conv_spec = np.sum([np.outer(a1[i], da1.W[:,oppo_list[i]]) for i in range(len(oppo_list))], axis=0)

In [ ]:
plt.plot(conv_spec[0])
plt.plot(conv_spec[1000])
plt.plot(conv_spec[9000])

In [ ]:
ce_dot = ((sy.T-conv_spec).reshape(9025,1,156)@(da1.W[:,endmember]-conv_spec).reshape(9025,156,1)).reshape(9025)

In [ ]:
recap = ((da1.W[:,0]-conv_spec).reshape(9025,1,156)@(da1.W[:,0]-conv_spec).reshape(9025,156,1)).reshape(9025)

In [ ]:
plt.imshow((ce_dot/recap).reshape(95,95))
plt.colorbar()

In [ ]:
plt.imshow(recap.reshape(95,95))
plt.colorbar()

In [ ]:
plt.imshow(ce_dot.reshape(95,95))
plt.colorbar()

In [ ]:
mu = 1

In [ ]:
y0.shape

In [ ]:
m_numerator = max_curve*y0-(6*(1-a3)*y0**2-6*(1+a2-2*a3)*y0+6*(a2-a3))
m_denominator = max_curve

In [ ]:
plt.imshow(m_numerator.reshape(95,95))
plt.colorbar()

In [ ]:
plt.imshow((ce_dot/recap).reshape(95,95))
plt.colorbar()

In [ ]:
est_a = lambda mu: np.maximum(((ce_dot + mu*m_numerator)/(recap + mu*m_denominator)),0)

In [ ]:
plt.imshow((est_a(0.1)-est_a(0.0)).reshape(95,95))
plt.colorbar()

In [ ]:
plt.imshow((est_a(.1)-est_a(0.0)).reshape(95,95))
plt.colorbar()